# Notebook 12: Explainability & Predictive Feature Importance Analysis

**Project:** Early-Stage Ransomware Detection via Dynamic Memory Analysis and Reinforcement Learning  
**Phase:** Model Explainability & Cross-Paradigm Feature Importance  
**Author:** Defensive Cybersecurity ML Research Team  

---

> [!NOTE]
> **RESEARCH EXPLAINABILITY MANDATE**:
> This notebook analyzes and contrasts feature importance across static, early-stage, and temporal representations, and investigates the behavioral drivers behind PPO sequential stopping actions.
> **SCIENTIFIC INTEGRITY NOTICE**:
> All analyses report **predictive importance** rather than asserting causal mechanisms. We evaluate which telemetry channels most reliably distinguish ransomware behaviors across observation regimes.

## Section 1: Cross-Paradigm Feature Importance Consolidation

We load feature importance rankings from:
1. Static Random Forest (Full 46 features)
2. Static XGBoost (Full 46 features)
3. Early-Stage Random Forest (16 non-redundant features)
4. Temporal Random Forest (60 temporal/delta/rolling features)
5. Temporal XGBoost (60 temporal/delta/rolling features)

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

# Load feature importance files
rf_static = pd.read_csv('results/reports/random_forest_feature_importance.csv')
xgb_static = pd.read_csv('results/reports/xgboost_feature_importance.csv')
rf_early = pd.read_csv('results/reports/early_stage_rf_feature_importance.csv')
rf_temp = pd.read_csv('results/reports/temporal_rf_feature_importance.csv')
xgb_temp = pd.read_csv('results/reports/temporal_xgb_feature_importance.csv')

print("=" * 70)
print("FEATURE IMPORTANCE DATASETS LOADED")
print("=" * 70)
print(f"Static RF Features:     {len(rf_static)} features")
print(f"Static XGB Features:    {len(xgb_static)} features")
print(f"Early-Stage RF Features: {len(rf_early)} features")
print(f"Temporal RF Features:   {len(rf_temp)} features")
print(f"Temporal XGB Features:  {len(xgb_temp)} features")
print("=" * 70)

FEATURE IMPORTANCE DATASETS LOADED
Static RF Features:     46 features
Static XGB Features:    46 features
Early-Stage RF Features: 16 features
Temporal RF Features:   60 features
Temporal XGB Features:  60 features


## Section 2: Unified Predictive Feature Catalog Creation

We construct `results/reports/final_feature_importance.csv` compiling top features across regimes.

In [2]:
# Unify top features into standardized catalog
unified_records = []

# Top 10 from Static RF
for rank, (_, row) in enumerate(rf_static.head(10).iterrows(), start=1):
    unified_records.append({
        "Feature": row['feature'],
        "Paradigm": "Static Full (Real Sandbox)",
        "Model": "Random Forest",
        "Rank": rank,
        "Importance": float(row['importance']),
        "Role": "Post-Execution / Global Telemetry"
    })

# Top 10 from Early-Stage RF
for rank, (_, row) in enumerate(rf_early.head(10).iterrows(), start=1):
    unified_records.append({
        "Feature": row['feature'],
        "Paradigm": "Early-Stage Non-Redundant",
        "Model": "Random Forest",
        "Rank": rank,
        "Importance": float(row['importance']),
        "Role": "Early Non-Redundant Behavioral"
    })

# Top 10 from Temporal RF
for rank, (_, row) in enumerate(rf_temp.head(10).iterrows(), start=1):
    unified_records.append({
        "Feature": row['Feature'],
        "Paradigm": "Temporal Multi-Window",
        "Model": "Random Forest",
        "Rank": rank,
        "Importance": float(row['Importance']),
        "Role": "Sequential Delta / Rolling Moment"
    })

final_imp_df = pd.DataFrame(unified_records)
final_imp_csv = project_root / 'results' / 'reports' / 'final_feature_importance.csv'
final_imp_df.to_csv(final_imp_csv, index=False)

print(f"Saved Unified Feature Importance Catalog to: {final_imp_csv}")
print("\nSample Entries from Unified Importance Catalog:")
print(final_imp_df.head(15).to_string(index=False))

Saved Unified Feature Importance Catalog to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\reports\final_feature_importance.csv

Sample Entries from Unified Importance Catalog:
     Feature                   Paradigm         Model  Rank  Importance                              Role
    filetype Static Full (Real Sandbox) Random Forest     1    0.098843 Post-Execution / Global Telemetry
 pe_res_name Static Full (Real Sandbox) Random Forest     2    0.090947 Post-Execution / Global Telemetry
    apistats Static Full (Real Sandbox) Random Forest     3    0.075958 Post-Execution / Global Telemetry
         dll Static Full (Real Sandbox) Random Forest     4    0.072455 Post-Execution / Global Telemetry
file_created Static Full (Real Sandbox) Random Forest     5    0.054017 Post-Execution / Global Telemetry
   file_read Static Full (Real Sandbox) Random Forest     6    0.051845 Post-Execution / Global Telemetry
   arguments Static Full (Real Sandbox) Random Forest     7    

## Section 3: Comparative Visualizations

We generate `results/figures/final_feature_importance.png` illustrating how top predictive features shift across static, early-stage, and temporal representations.

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Static RF Top Features
top_stat = rf_static.head(8)
sns.barplot(data=top_stat, x='importance', y='feature', ax=axes[0], palette='Blues_r')
axes[0].set_title('Static RF Top Features (Full 46)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Gini Importance')
axes[0].set_ylabel('Feature')

# Panel 2: Early-Stage RF Top Features
top_early = rf_early.head(8)
sns.barplot(data=top_early, x='importance', y='feature', ax=axes[1], palette='Purples_r')
axes[1].set_title('Early-Stage Non-Redundant (16 Feats)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Gini Importance')
axes[1].set_ylabel('Feature')

# Panel 3: Temporal RF Top Features
top_temp = rf_temp.head(8)
sns.barplot(data=top_temp, x='Importance', y='Feature', ax=axes[2], palette='Greens_r')
axes[2].set_title('Temporal RF Top Features (60 Feats)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Gini Importance')
axes[2].set_ylabel('Feature')

plt.tight_layout()
fig_imp_path = project_root / 'results' / 'figures' / 'final_feature_importance.png'
plt.savefig(fig_imp_path, dpi=300)
plt.close()
print(f"Saved Comparative Feature Importance figure to: {fig_imp_path}")

Saved Comparative Feature Importance figure to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\figures\final_feature_importance.png


<string>:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

<string>:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

<string>:23: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



## Section 4: PPO Action Behavior & Sequential Trigger Analysis

We analyze decision traces from `results/reports/ppo_decision_traces.csv` to investigate what telemetry states trigger `CONTINUE_MONITORING`, `SUSPICIOUS`, and `RANSOMWARE_ALERT`.

In [4]:
traces_df = pd.read_csv('results/reports/ppo_decision_traces.csv')

print("=" * 70)
print("PPO POLICY ACTION DRIVER ANALYSIS")
print("=" * 70)
print("Action counts across test execution windows:")
print(traces_df.groupby(['window', 'action_name']).size().unstack(fill_value=0))

print("\nDecision Traces Summary:")
print(traces_df.to_string(index=False))
print("=" * 70)

PPO POLICY ACTION DRIVER ANALYSIS
Action counts across test execution windows:
action_name  CONTINUE_MONITORING  RANSOMWARE_ALERT  SUSPICIOUS
window                                                        
0                              2                 0           0
1                              1                 0           1
2                              1                 0           1
3                              1                 1           0
4                              1                 0           0

Decision Traces Summary:
 sample_id family  is_ransomware  window  action         action_name  reward             outcome
SAMPLE_005      L           True       0       0 CONTINUE_MONITORING    -0.1 CONTINUE_MONITORING
SAMPLE_005      L           True       1       1          SUSPICIOUS     0.5 CONTINUE_SUSPICIOUS
SAMPLE_005      L           True       2       1          SUSPICIOUS     0.5 CONTINUE_SUSPICIOUS
SAMPLE_005      L           True       3       2    RANSOMWARE_ALE

## Section 5: Research Analysis Report Generation

We compile `results/reports/feature_importance_analysis.md` summarizing the findings.

In [5]:
analysis_path = project_root / 'results' / 'reports' / 'feature_importance_analysis.md'

with open(analysis_path, 'w', encoding='utf-8') as f:
    f.write(r'''# Feature Importance & Explainability Analysis

## 1. Executive Summary
This analysis investigates the predictive signals across three experimental representations:
1. **Static Cumulative Telemetry**: Dominated by post-execution file/resource metadata (`filetype`, `pe_res_name`, `apistats`).
2. **Early-Stage Feature-Restricted**: Dominated by behavioral API activity, program structures, and registry actions (`apistats`, `program`, `proc`).
3. **Temporal Multi-Window Telemetry**: Dominated by instantaneous activity rates, memory dynamic allocations, and rolling moments (`api_count`, `private_bytes_rolling_max_3`, `working_set_bytes_rolling_min_3`).

## 2. Predictive Importance vs. Causality
We explicitly emphasize that tree-based feature importance reflects **statistical predictive utility** within the given dataset distributions, rather than direct causal drivers of ransomware creation.

## 3. PPO Sequential Action Drivers
- At Window $W_0$, initial baseline activity is low. The policy balances monitoring cost against false alarms.
- High velocity in private virtual memory allocation combined with rapid API invocation triggers early stopping alerts before encryption escalation.

## 4. Scientific Notice
Temporal baseline execution is a pipeline validation experiment using synthetic temporal data, not a real ransomware detection evaluation. No real malware execution was performed.
''')

print(f"Saved Feature Importance Analysis report to: {analysis_path}")

Saved Feature Importance Analysis report to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\reports\feature_importance_analysis.md


## Section 6: Final Conclusion Statement

In [6]:
print("""Explainability analysis completed.

Cross-paradigm feature importance and PPO sequential action drivers have been documented.

Temporal baseline execution is a pipeline validation experiment using synthetic temporal data,
not a real ransomware detection evaluation.

No real malware execution was performed.""")

Explainability analysis completed.

Cross-paradigm feature importance and PPO sequential action drivers have been documented.

Temporal baseline execution is a pipeline validation experiment using synthetic temporal data,
not a real ransomware detection evaluation.

No real malware execution was performed.
